# Architecture sweep over the calculated bottlenecks


In [ ]:
import contextlib
import io
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import accelforge as af


In [ ]:
_models = Path("../models")
HYBRID_SWEEP_ARCH = _models / "tpu_aim_hybrid_sweep.yaml"
_Q_WL = _models / "gpt3_6.7B_kv_cache.yaml"

_Q_PARAMS_BASE = {
    "AIM_BUFFER_KB": 2,
    "HBM_INTERNAL_GBPS": 512,
}
PU_FANOUT_SWEEP = [16, 24, 32, 64, 128]
HBM_EXTERNAL_SWEEP = [8, 16, 32, 64, 128]
_FIXED_EXT_FOR_FANOUT_SWEEP = 32
_FIXED_FANOUT_FOR_EXT_SWEEP = 16
N_TOKENS = 8196
BATCH_SIZE = 1
N_NEW_TOKENS = 1
# Same sequence length for both phases (typical: prompt length = cache length).
_CTX = int(N_TOKENS)

In [ ]:
def _pick_edp(mapping):
    energies = mapping.energy()
    latencies = mapping.latency()
    if isinstance(energies, list):
        idx = min(range(len(energies)), key=lambda i: energies[i] * latencies[i])
        e, l = float(energies[idx]), float(latencies[idx])
    else:
        e, l = float(energies), float(latencies)
    return e, l, e * l


def _q_new_params(
    aim_fanout: int,
    hbm_external: int,
    *,
    n_tokens: int,
    n_new_tokens: int,
) -> dict:
    return {
        "BATCH_SIZE": BATCH_SIZE,
        "N_TOKENS": n_tokens,
        "N_NEW_TOKENS": n_new_tokens,
        **_Q_PARAMS_BASE,
        "AIM_PU_FANOUT": aim_fanout,
        "HBM_EXTERNAL_GBPS": hbm_external,
    }


def _aim_area_mm2(params: dict) -> float:
    spec = af.Spec.from_yaml(str(HYBRID_SWEEP_ARCH), str(_Q_WL), jinja_parse_data=params)
    spec = spec.calculate_component_area_energy_latency_leak()
    per_comp = dict(spec.arch.per_component_total_area)
    return  per_comp.get("PU_MAC", 0.0)


def _map_q_new(
    aim_fanout: int,
    hbm_external: int,
    *,
    n_tokens: int,
    n_new_tokens: int,
) -> tuple[float, float, float]:
    params = _q_new_params(
        aim_fanout,
        hbm_external,
        n_tokens=n_tokens,
        n_new_tokens=n_new_tokens,
    )
    spec = af.Spec.from_yaml(
        str(HYBRID_SWEEP_ARCH), str(_Q_WL), jinja_parse_data=params
    )
    spec.mapper.metrics = af.Metrics.LATENCY | af.Metrics.ENERGY
    with contextlib.redirect_stdout(io.StringIO()):
        m = spec.map_workload_to_arch(einsum_names=["Q_new"], print_progress=False)
    return _pick_edp(m)


In [ ]:
# Decode: one new token, full KV length M_FULL = _CTX
rows_fanout = []
for f in PU_FANOUT_SWEEP:
    params = _q_new_params(
        f,
        _FIXED_EXT_FOR_FANOUT_SWEEP,
        n_tokens=_CTX,
        n_new_tokens=1,
    )
    e, l, edp = _map_q_new(
        f,
        _FIXED_EXT_FOR_FANOUT_SWEEP,
        n_tokens=_CTX,
        n_new_tokens=1,
    )
    rows_fanout.append({
        "aim_fanout": f,
        "aim_area_mm2": _aim_area_mm2(params),
        "energy_j": e,
        "latency_s": l,
        "edp_j_s": edp,
    })

# Prefill: all prompt tokens at once (M = M_FULL = _CTX)
rows_ext = []
for g in HBM_EXTERNAL_SWEEP:
    params = _q_new_params(
        _FIXED_FANOUT_FOR_EXT_SWEEP,
        g,
        n_tokens=_CTX,
        n_new_tokens=_CTX,
    )
    e, l, edp = _map_q_new(
        _FIXED_FANOUT_FOR_EXT_SWEEP,
        g,
        n_tokens=_CTX,
        n_new_tokens=_CTX,
    )
    rows_ext.append({
        "hbm_external_gbps": g,
        "aim_area_mm2": _aim_area_mm2(params),
        "energy_j": e,
        "latency_s": l,
        "edp_j_s": edp,
    })

df_fan = pd.DataFrame(rows_fanout)
df_ext = pd.DataFrame(rows_ext)
df_fan["edap_j_s_mm2"] = df_fan["edp_j_s"] * df_fan["aim_area_mm2"]
df_ext["edap_j_s_mm2"] = df_ext["edp_j_s"] * df_ext["aim_area_mm2"]

best_fan = df_fan.loc[df_fan["edap_j_s_mm2"].idxmin()]
best_ext = df_ext.loc[df_ext["edap_j_s_mm2"].idxmin()]
print("EDAP-optimal Q_new decode PU-fanout point:")
print(
    f"  fanout={int(best_fan.aim_fanout)} ({int(best_fan.aim_fanout**2)} PUs), "
    f"area={best_fan.aim_area_mm2:.3e} mm², "
    f"E={best_fan.energy_j:.3e} J, L={best_fan.latency_s:.3e} s, "
    f"EDAP={best_fan.edap_j_s_mm2:.3e} J*s*mm²"
)
print("EDAP-optimal Q_new prefill HBM-external point:")
print(
    f"  external={int(best_ext.hbm_external_gbps)} GB/s, "
    f"area={best_ext.aim_area_mm2:.3e} mm², "
    f"E={best_ext.energy_j:.3e} J, L={best_ext.latency_s:.3e} s, "
    f"EDAP={best_ext.edap_j_s_mm2:.3e} J*s*mm²"
)

fig, (axL, axR) = plt.subplots(1, 2, figsize=(13, 4.5))

(l_e,) = axL.plot(df_fan["aim_fanout"], df_fan["energy_j"], marker="o", color="C0", label="Energy (J)")
axL.set_xlabel("AIM PU fanout (side length)")
axL.set_ylabel("Energy (J)", color="C0")
axL.tick_params(axis="y", labelcolor="C0")
axL_t = axL.twinx()
(l_l,) = axL_t.plot(df_fan["aim_fanout"], df_fan["latency_s"], marker="s", color="C1", label="Latency (s)")
axL_t.set_ylabel("Latency (s)", color="C1")
axL_t.tick_params(axis="y", labelcolor="C1")

# Overlay AiM area as histogram-style bars on the same fanout x-axis.
axL_area = axL.twinx()
axL_area.spines["right"].set_position(("outward", 52))
bar_width = 0.65 * min(np.diff(sorted(df_fan["aim_fanout"])))
area_bars = axL_area.bar(
    df_fan["aim_fanout"],
    df_fan["aim_area_mm2"],
    width=bar_width,
    color="0.5",
    alpha=0.18,
    label="AiM area (mm²)",
    zorder=0,
)
axL_area.set_ylabel("")
axL_area.tick_params(axis="y", labelright=False, right=False)
axL_area.spines["right"].set_visible(False)
# axL.set_zorder(axL_area.get_zorder() + 1)
# axL.patch.set_visible(False)

axL.set_title(
    f"Q_new decode — sweep AIM PU fanout\n"
    f"(M=1, M_FULL={_CTX}; OffChipLink ext = {_FIXED_EXT_FOR_FANOUT_SWEEP} GB/s)"
)
axL.grid(True, alpha=0.3)
axL.legend(handles=[l_e, l_l, area_bars], loc="best", fontsize=8)

(l_e2,) = axR.plot(df_ext["hbm_external_gbps"], df_ext["energy_j"], marker="o", color="C0", label="Energy (J)")
axR.set_xlabel("HBM external bandwidth (GB/s) — TPU OffChipLink")
axR.set_ylabel("Energy (J)", color="C0")
# Set the y range so the values fall in the middle and each tick is 1
energy_vals = df_ext["energy_j"]
ymin = np.floor(energy_vals.min()) - 1
ymax = np.ceil(energy_vals.max()) + 1
axR.set_ylim([ymin, ymax])
axR.set_yticks(np.arange(ymin, ymax + 1, 1))
axR.tick_params(axis="y", labelcolor="C0")
axR_t = axR.twinx()
(l_l2,) = axR_t.plot(df_ext["hbm_external_gbps"], df_ext["latency_s"], marker="s", color="C1", label="Latency (s)")
axR_t.set_ylabel("Latency (s)", color="C1")
axR_t.tick_params(axis="y", labelcolor="C1")
axR.set_title(
    f"Q_new prefill — sweep TPU OffChipLink (HBM external)\n"
    f"(M=M_FULL={_CTX}; AIM fanout = {_FIXED_FANOUT_FOR_EXT_SWEEP})"
)
axR.grid(True, alpha=0.3)
axR.legend(handles=[l_e2, l_l2], loc="best", fontsize=8)

plt.tight_layout()
plt.show()
